### IMPORT

In [ ]:
import numpy as np
import pandas as pd

raw_data = pd.read_csv('../data/employee_attrition_2026_master.csv')


### Checking Missing / Duplicate / raw_data Type

In [ ]:
raw_data.head()

In [ ]:
raw_data.info()

print(f"duplicated = {raw_data.duplicated().sum()}")
print(f"size = {raw_data.size}")
print(f"shape = {raw_data.shape}\n")

### Claening

* Drop columns
    * employee_id ตัวชี้วัดไม่ได้ใช้  
    * attrition_reason ไม่เกี่ยวกับการคำนวณ   
    * is_manager ซ้ำซ้อนกับ job_level   
    * generation ตัวแปรอิสระสัมพันธ์กันเองสูงเกินไปกับ *age*  
*  กรองข้อมูลผิดปกติที่ 
    * อายุการทำงาน และอายุ ไม่สมเหตุสผลกัน ตัวอย่าง : age 20, tenure 9.8 years = เริ่มทำงานที่อายุ 10.2 ปี จึงทำการกรองข้อมูลออก
* นับจำนวน
    * จำนวนแถวทั้งหมด = 45,000 row
    * ถูกคัดกรองออกจำนวน 1,561 row จากเงื่อนไขอายุเริ่มทำงานต้องไม่ต่ำกว่า 20 ปี  
    คิดเป็น 3.469 % จากข้อมูลทั้งหมด

In [ ]:
# Drop columns
raw_data = raw_data.drop(columns=["employee_id", "attrition_reason", "is_manager", "generation"])

# กรองข้อมูลผิดปกติที่ อายุการทำงาน และอายุ ไม่สมเหตุสผลกัน
bad_mask = raw_data["tenure_years"] > (raw_data["age"] - 16)
n_bad = bad_mask.sum()

# check removed rows
print(f"rows to remove = {n_bad} \n45,000-1,561 = {45000-n_bad} \nคิดเป็น = {(n_bad/45000)*100:.3f} % ")

clean_data = raw_data[~bad_mask].reset_index(drop=True)

### Outlier

In [ ]:
# 1. เช็ค outlier ปกติทางสถิติด้วย IQR (continuous columns only)
cols = ["age", "tenure_years", "salary_usd", "commute_minutes", "weekly_hours"]

for col in cols:
    q1 = clean_data[col].quantile(0.25)
    q3 = clean_data[col].quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    
    outliers = ~clean_data[col].between(lo, hi)
    n_outliers = outliers.sum()
    print(f"{col:16s} bounds=({lo:.1f}, {hi:.1f}) outliers={n_outliers} max={clean_data[col].max()}")

# 2. เช็ค work_arrangement vs days_in_office_required
print("\nwork_arrangement vs days_in_office_required:")
print(pd.crosstab(clean_data["work_arrangement"], clean_data["days_in_office_required"]))

# 3. เช็คช่วงเงินเดือนแยกตาม job level
print("\nSalary range by job level:")
print(clean_data.groupby("job_level")["salary_usd"].agg(["min", "max"]))

# 4. CHANGED: real salary outliers = far from own job level (|z| > 3)
salary_z = clean_data.groupby("job_level")["salary_usd"].transform(lambda s: (s - s.mean()) / s.std())
print(f"\nSalary outliers inside own job level (|z| > 3): {(salary_z.abs() > 3).sum()}")

In [ ]:
# Log transform: shrink long tail, keep order, keep all rows
clean_data["tenure_years"] = np.log1p(clean_data["tenure_years"])
clean_data["commute_minutes"] = np.log1p(clean_data["commute_minutes"])

# Salary compared with own job level
# คำนวณ Median ตามระดับงาน แล้วสร้างคอลัมน์ใหม่
level_median = clean_data.groupby("job_level")["salary_usd"].transform("median")
clean_data["salary_vs_level"] = clean_data["salary_usd"] / level_median

print(f"Done, shape: {clean_data.shape}")

In [ ]:
# # Log transform: shrink long tail, keep order, keep all rows
# clean_data["tenure_years"] = np.log1p(clean_data["tenure_years"])
# clean_data["commute_minutes"] = np.log1p(clean_data["commute_minutes"])

# # Keep salary_usd unchanged

# # คำนวณ Median ตามระดับงาน แล้วสร้างคอลัมน์ใหม่
# median_by_level = clean_data.groupby('job_level')['salary_usd'].transform('median')
# clean_data['salary_vs_level'] = clean_data['salary_usd'] / median_by_level

# print(f"Done, shape: {clean_data.shape}") ....

In [ ]:
# Verify
print(clean_data[["tenure_years", "commute_minutes", "salary_vs_level"]].describe().loc[["min", "max"]])

# # salary_vs_level not same info as salary_vs_market_pct (corr ~ 0) → keep both
# print(clean_data[["salary_vs_level", "salary_vs_market_pct"]].corr())

In [ ]:
clean_data.info()

> Strict note: `level_median` uses all rows. Tiny leak with 43k rows. Strict version → split train/test first, compute `level_median` from train only, apply to test.

# encoding

|  | Column | Encoding Method | Reason |
| :--- | :--- | :--- | :--- |
| 1 | job_level | OrdinalEncoder | Has order rank : Junior < Mid < Senior < Lead < Manager < Director |
| 2 | ai_tools_adoption | OrdinalEncoder | Has natural rank : none < light < regular < power |
| 3 | gender | OneHotEncoder | Nominal, no order prevents the model from assuming mathematical ranking|
| 4 | department | OneHotEncoder | Nominal, no order prevents the model from assuming mathematical ranking|
| 5 | work_arrangement | OneHotEncoder | Nominal, no order prevents the model from assuming mathematical ranking|

In [ ]:
# Ordinal encoding
job_level_order = ["Junior", "Mid", "Senior", "Lead", "Manager", "Director"]
clean_data["job_level"] = pd.Categorical(clean_data["job_level"], categories=job_level_order, ordered=True).codes
assert (clean_data["job_level"] >= 0).all()   # -1 = value not in list
clean_data["job_level"].value_counts().sort_index()

In [ ]:
# Ordinal encoding
ai_order = ["none", "light", "regular", "power"]
clean_data["ai_tools_adoption"] = pd.Categorical(clean_data["ai_tools_adoption"], categories=ai_order, ordered=True).codes
assert (clean_data["ai_tools_adoption"] >= 0).all()   # -1 = value not in list
clean_data["ai_tools_adoption"].value_counts().sort_index()

In [ ]:
# One-hot encoding
nominal_cols = ["gender", "department", "work_arrangement"]
clean_data = pd.get_dummies(clean_data, columns=nominal_cols, drop_first=False, dtype=int)

In [ ]:
clean_data.info()

In [ ]:
clean_data.to_csv("data_preprocess2.csv", index=False)

## Summary
---
### Data Preprocessing Summary
* **Initial Data** : 45,000 rows, 29 columns
* **Cleaned Data** : 43,439 rows, 37 columns

#### 1. Data Cleaning
* **Dropped Columns** : employee_id, attrition_reason, is_manager, generation
* **Logic Error Removed** : records where tenure_years > (age - 16) (1,561 rows)
* **Outlier Handling** :
  * tenure_years, commute_minutes → np.log1p (no row removed, no clip)
  * salary_usd → replaced by salary_vs_level (salary ÷ median of own job level)
  * age, weekly_hours → kept (values possible)

#### 2. Feature Encoding
* **Ordinal Encoding** :
  * job_level: Junior < Mid < Senior < Lead < Manager < Director
  * ai_tools_adoption: none < light < regular < power
* **One-Hot Encoding (drop_first=True)** : gender, department, work_arrangement

In [ ]:
clean_data.info()